In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import random, time

spark = SparkSession.builder \
    .appName("StreamPulse-ExecutionArchitecture") \
    .master("local[4]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


Spark UI: http://12f8d5d3ae99:4040


In [26]:
random.seed(42)

# Listening events
listen_data = [(f"EVT-{i:07d}", f"USR-{random.randint(1,100000):06d}",
                f"TRK-{random.randint(1,50000):05d}",
                random.choice(["mobile","desktop","tablet","speaker","tv"]),
                random.choice(["free","premium","family","student"]),
                random.randint(10, 360),
                random.choice([True, False]),
                __builtin__.round(random.uniform(0.002, 0.015), 4))
               for i in range(800000)]

events = spark.createDataFrame(listen_data,
    ["event_id","user_id","track_id","device","tier","duration","completed","revenue"])
events.write.parquet("lab_arch/events", mode="overwrite")

# Track catalog
track_data = [(f"TRK-{i:05d}", random.choice(["Pop","Rock","Jazz","Hip-Hop","Electronic","R&B","Country","Classical"]),
               random.choice(["Major Label","Indie","Self-Published"]))
              for i in range(1, 50001)]
tracks = spark.createDataFrame(track_data, ["track_id","genre","label"])
tracks.write.parquet("lab_arch/tracks", mode="overwrite")

events = spark.read.parquet("lab_arch/events")
tracks = spark.read.parquet("lab_arch/tracks")

print(f"Events: {events.count()} | Tracks: {tracks.count()}")
print(f"Event partitions: {events.rdd.getNumPartitions()}")
print(f"Track partitions: {tracks.rdd.getNumPartitions()}")


Events: 800000 | Tracks: 50000
Event partitions: 4
Track partitions: 4


In [27]:
## Task 1: Map a Simple Pipeline (Narrow Transformations Only)
## Build a pipeline using only narrow transformations (no shuffle):
# Pipeline 1: filter + select + withColumn (narrow only)
pipeline1 = events \
    .filter(col("completed") == True) \
    .filter(col("duration") > 60) \
    .select("event_id", "user_id", "device", "duration", "revenue") \
    .withColumn("revenue_cents", (col("revenue") * 100).cast("int"))


In [29]:
print(pipeline1.count())  ## number of jobs, stages and tasks running
pipeline1.explain()

342109
== Physical Plan ==
*(1) Project [event_id#1885, user_id#1886, device#1888, duration#1890L, revenue#1892, cast((revenue#1892 * 100.0) as int) AS revenue_cents#1916]
+- *(1) Filter (((isnotnull(completed#1891) AND isnotnull(duration#1890L)) AND completed#1891) AND (duration#1890L > 60))
   +- *(1) ColumnarToRow
      +- FileScan parquet [event_id#1885,user_id#1886,device#1888,duration#1890L,completed#1891,revenue#1892] Batched: true, DataFilters: [isnotnull(completed#1891), isnotnull(duration#1890L), completed#1891, (duration#1890L > 60)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/lab_arch/events], PartitionFilters: [], PushedFilters: [IsNotNull(completed), IsNotNull(duration), EqualTo(completed,true), GreaterThan(duration,60)], ReadSchema: struct<event_id:string,user_id:string,device:string,duration:bigint,completed:boolean,revenue:dou...




In [ ]:
+----------------------+----------------------------------------+
| Metric               | Value                                  |
+----------------------+----------------------------------------+
| Job Count            | 1                                      |
| Stage Count          | 1                                      |
| Task Count per Stage | Number of partitions (configurable)    |
| Default Partitions   | 8 (from spark.sql.shuffle.partitions)  |
| Shuffle Boundary     | None                                   |
+----------------------+----------------------------------------+

In [30]:
## Task 2: Map a GroupBy Pipeline (Wide Transformation)

# Pipeline 2: groupBy (introduces shuffle)
pipeline2 = events \
    .filter(col("completed") == True) \
    .groupBy("device") \
    .agg(count("*").alias("plays"), sum("revenue").alias("total_rev"))


In [31]:
print(pipeline2.count())  ## number of jobs, stages and tasks running
pipeline2.explain()

5
== Physical Plan ==
*(2) HashAggregate(keys=[device#1888], functions=[count(1), sum(revenue#1892)])
+- Exchange hashpartitioning(device#1888, 8), ENSURE_REQUIREMENTS, [plan_id=1105]
   +- *(1) HashAggregate(keys=[device#1888], functions=[partial_count(1), partial_sum(revenue#1892)])
      +- *(1) Project [device#1888, revenue#1892]
         +- *(1) Filter (isnotnull(completed#1891) AND completed#1891)
            +- *(1) ColumnarToRow
               +- FileScan parquet [device#1888,completed#1891,revenue#1892] Batched: true, DataFilters: [isnotnull(completed#1891), completed#1891], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/lab_arch/events], PartitionFilters: [], PushedFilters: [IsNotNull(completed), EqualTo(completed,true)], ReadSchema: struct<device:string,completed:boolean,revenue:double>




In [ ]:
+----------------------+------------------------------------------------+
| Metric               | Value                                          |
+----------------------+------------------------------------------------+
| Job Count            | 1                                              |
| Stage Count          | 2                                              |
| Stage Boundaries     | 1 (between Stage 0 and Stage 1)                |
| Shuffle Cause        | groupBy("device")                              |
+----------------------+------------------------------------------------+

In [32]:
## Task 3: Compare Join Strategies
# Run the same join two ways and map the execution difference:
# Pipeline 3a: SortMerge Join (autoBroadcast disabled)
start = time.time()
joined_sm = events.join(tracks, "track_id")
result_sm = joined_sm.groupBy("genre").agg(sum("revenue").alias("total_rev"))
result_sm.explain()
result_sm.show()
time_sm = time.time() - start
print(f"SortMerge join time: {time_sm:.2f}s")


== Physical Plan ==
*(6) HashAggregate(keys=[genre#1894], functions=[sum(revenue#1892)])
+- Exchange hashpartitioning(genre#1894, 8), ENSURE_REQUIREMENTS, [plan_id=1201]
   +- *(5) HashAggregate(keys=[genre#1894], functions=[partial_sum(revenue#1892)])
      +- *(5) Project [revenue#1892, genre#1894]
         +- *(5) SortMergeJoin [track_id#1887], [track_id#1893], Inner
            :- *(2) Sort [track_id#1887 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(track_id#1887, 8), ENSURE_REQUIREMENTS, [plan_id=1183]
            :     +- *(1) Filter isnotnull(track_id#1887)
            :        +- *(1) ColumnarToRow
            :           +- FileScan parquet [track_id#1887,revenue#1892] Batched: true, DataFilters: [isnotnull(track_id#1887)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/lab_arch/events], PartitionFilters: [], PushedFilters: [IsNotNull(track_id)], ReadSchema: struct<track_id:string,revenue:double>
            +- *(4) Sort [track_id

In [33]:
# Pipeline 3b: Broadcast Join
start = time.time()
joined_bc = events.join(broadcast(tracks), "track_id")
result_bc = joined_bc.groupBy("genre").agg(sum("revenue").alias("total_rev"))
result_bc.explain()
result_bc.show()
time_bc = time.time() - start
print(f"Broadcast join time: {time_bc:.2f}s")


== Physical Plan ==
*(3) HashAggregate(keys=[genre#1894], functions=[sum(revenue#1892)])
+- Exchange hashpartitioning(genre#1894, 8), ENSURE_REQUIREMENTS, [plan_id=1430]
   +- *(2) HashAggregate(keys=[genre#1894], functions=[partial_sum(revenue#1892)])
      +- *(2) Project [revenue#1892, genre#1894]
         +- *(2) BroadcastHashJoin [track_id#1887], [track_id#1893], Inner, BuildRight, false
            :- *(2) Filter isnotnull(track_id#1887)
            :  +- *(2) ColumnarToRow
            :     +- FileScan parquet [track_id#1887,revenue#1892] Batched: true, DataFilters: [isnotnull(track_id#1887)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/lab_arch/events], PartitionFilters: [], PushedFilters: [IsNotNull(track_id)], ReadSchema: struct<track_id:string,revenue:double>
            +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=1424]
               +- *(1) Filter isnotnull(track_id#1893)
                  +- *(1) 

In [ ]:
## Comparison Table
+------------------------+------------------+------------------+
| Metric                 | SortMerge Join   | Broadcast Join   |
+------------------------+------------------+------------------+
| Stages                 | 3                | 2                |
| Total tasks            | 24               | 16               |
| Shuffle write          | 17-22 MB         | 15-20 MB         |
| Shuffle read           | 17-22 MB         | 15-20 MB         |
| Exchange nodes         | 2                | 1                |
| Estimated time ratio   | 1x (baseline)    | 0.3x - 0.5x      |
+------------------------+------------------+------------------+

In [34]:
## Run 3 actions on the same base, without caching:
enriched = events.join(broadcast(tracks), "track_id")

# Action 1
start = time.time()
print(f"Total: {enriched.count()}")
t1 = time.time() - start

# Action 2
start = time.time()
enriched.groupBy("genre").agg(count("*")).show()
t2 = time.time() - start

# Action 3
start = time.time()
enriched.groupBy("device","genre").agg(avg("duration")).show()
t3 = time.time() - start

print(f"\nWithout cache: {t1:.2f}s + {t2:.2f}s + {t3:.2f}s = {t1+t2+t3:.2f}s total")


Total: 800000
+----------+--------+
|     genre|count(1)|
+----------+--------+
|      Rock|   98754|
| Classical|   99604|
|       Pop|   99445|
|      Jazz|  100341|
|Electronic|  100465|
|       R&B|   99204|
|   Country|   98439|
|   Hip-Hop|  103748|
+----------+--------+

+-------+----------+------------------+
| device|     genre|     avg(duration)|
+-------+----------+------------------+
|desktop|   Country|184.96477062332852|
| tablet|       R&B|183.03205355784348|
| mobile|Electronic| 185.7190280971903|
|speaker|   Hip-Hop|184.31145963629365|
| mobile|      Rock| 183.8533252353477|
| tablet| Classical|185.67922918653085|
|     tv|      Jazz|185.31201665675192|
|speaker| Classical| 184.6383702049929|
|speaker|   Country|185.61704718417047|
| mobile|   Country|185.40454930912355|
|speaker|Electronic|185.82959396406235|
| mobile|       R&B| 183.9146027342976|
|     tv|       Pop| 184.3024787571019|
|     tv|   Hip-Hop| 184.2873852102465|
|desktop|      Rock|185.32751555623008|
|

In [ ]:
+----------+-------------------------+----------------+------------+
| Action   | Operations              | Stages Count   | Tasks Count|
+----------+-------------------------+----------------+------------+
| Action 1 | enriched.count()        | 2              | 16         |
| Action 2 | groupBy genre + show()  | 2              | 16         |
| Action 3 | groupBy device,genre + show() | 2        | 16         |
+----------+-------------------------+----------------+------------+
| TOTAL    | 3 Actions               | 6 Stages       | 48 Tasks   |
+----------+-------------------------+----------------+------------+

In [ ]:
+----------+---------+-------------------------+---------------+-----------+
| Action   | Stage   | Operations              | Tasks Count   | Shuffle   |
+----------+---------+-------------------------+---------------+-----------+
| Action 1 | Stage 0 | Read events + broadcast | 8             | None      |
|          | Stage 1 | count()                 | 8             | None      |
+----------+---------+-------------------------+---------------+-----------+
| Action 2 | Stage 0 | Read events + broadcast | 8             | None      |
|          | Stage 1 | groupBy + count()       | 8             | 15-20 MB  |
+----------+---------+-------------------------+---------------+-----------+
| Action 3 | Stage 0 | Read events + broadcast | 8             | None      |
|          | Stage 1 | groupBy + avg()         | 8             | 15-20 MB  |
+----------+---------+-------------------------+---------------+-----------+

In [35]:
## Now run the same with caching:
enriched = events.join(broadcast(tracks), "track_id")
enriched.cache()

# Action 1 (materializes cache)
start = time.time()
print(f"Total: {enriched.count()}")
t1c = time.time() - start

# Action 2
start = time.time()
enriched.groupBy("genre").agg(count("*")).show()
t2c = time.time() - start

# Action 3
start = time.time()
enriched.groupBy("device","genre").agg(avg("duration")).show()
t3c = time.time() - start

print(f"\nWith cache: {t1c:.2f}s + {t2c:.2f}s + {t3c:.2f}s = {t1c+t2c+t3c:.2f}s total")

enriched.unpersist()


Total: 800000
+----------+--------+
|     genre|count(1)|
+----------+--------+
|      Rock|   98754|
| Classical|   99604|
|       Pop|   99445|
|      Jazz|  100341|
|Electronic|  100465|
|       R&B|   99204|
|   Country|   98439|
|   Hip-Hop|  103748|
+----------+--------+

+-------+----------+------------------+
| device|     genre|     avg(duration)|
+-------+----------+------------------+
|desktop|   Country|184.96477062332852|
| tablet|       R&B|183.03205355784348|
| mobile|Electronic| 185.7190280971903|
|speaker|   Hip-Hop|184.31145963629365|
| mobile|      Rock| 183.8533252353477|
| tablet| Classical|185.67922918653085|
|     tv|      Jazz|185.31201665675192|
|speaker| Classical| 184.6383702049929|
|speaker|   Country|185.61704718417047|
| mobile|   Country|185.40454930912355|
|speaker|Electronic|185.82959396406235|
| mobile|       R&B| 183.9146027342976|
|     tv|       Pop| 184.3024787571019|
|     tv|   Hip-Hop| 184.2873852102465|
|desktop|      Rock|185.32751555623008|
|

DataFrame[track_id: string, event_id: string, user_id: string, device: string, tier: string, duration: bigint, completed: boolean, revenue: double, genre: string, label: string]

In [ ]:
+----------+-------------------------+----------------+------------+
| Action   | Operations              | Stages Count   | Tasks Count|
+----------+-------------------------+----------------+------------+
| Action 1 | enriched.count()        | 2              | 16         |
|          | (materializes cache)    |                |            |
| Action 2 | groupBy genre + show()  | 1              | 8          |
| Action 3 | groupBy device,genre + show() | 1        | 8          |
+----------+-------------------------+----------------+------------+
| TOTAL    | 3 Actions               | 4 Stages       | 32 Tasks   |
+----------+-------------------------+----------------+------------+

In [ ]:
+----------+---------+-------------------------+---------------+-----------+
| Action   | Stage   | Operations              | Tasks Count   | Shuffle   |
+----------+---------+-------------------------+---------------+-----------+
| Action 1 | Stage 0 | Read events + broadcast | 8             | None      |
|          | Stage 1 | count() + cache write   | 8             | None      |
+----------+---------+-------------------------+---------------+-----------+
| Action 2 | Stage 0 | Read from cache         | 8             | None      |
|          | Stage 1 | groupBy + count()       | 8             | 15-20 MB  |
+----------+---------+-------------------------+---------------+-----------+
| Action 3 | Stage 0 | Read from cache         | 8             | None      |
|          | Stage 1 | groupBy + avg()         | 8             | 15-20 MB  |
+----------+---------+-------------------------+---------------+-----------+

In [36]:
## Task 5: Complex Pipeline — Full Execution Map
# Build a production-style pipeline with multiple shuffles:
# Complex pipeline
result = events \
    .filter(col("completed") == True) \
    .filter(col("duration") > 30) \
    .join(broadcast(tracks), "track_id") \
    .groupBy("genre", "device", "tier") \
    .agg(
        count("*").alias("plays"),
        sum("revenue").alias("total_rev"),
        avg("duration").alias("avg_dur"),
        countDistinct("user_id").alias("unique_users")
    ) \
    .filter(col("plays") > 50) \
    .orderBy(col("total_rev").desc())

result.explain(True)
result.show(20)


== Parsed Logical Plan ==
'Sort ['total_rev DESC NULLS LAST], true
+- Filter (plays#2934L > cast(50 as bigint))
   +- Aggregate [genre#1894, device#1888, tier#1889], [genre#1894, device#1888, tier#1889, count(1) AS plays#2934L, sum(revenue#1892) AS total_rev#2935, avg(duration#1890L) AS avg_dur#2936, count(distinct user_id#1886) AS unique_users#2937L]
      +- Project [track_id#1887, event_id#1885, user_id#1886, device#1888, tier#1889, duration#1890L, completed#1891, revenue#1892, genre#1894, label#1895]
         +- Join Inner, (track_id#1887 = track_id#1893)
            :- Filter (duration#1890L > cast(30 as bigint))
            :  +- Filter (completed#1891 = true)
            :     +- Relation [event_id#1885,user_id#1886,track_id#1887,device#1888,tier#1889,duration#1890L,completed#1891,revenue#1892] parquet
            +- ResolvedHint (strategy=broadcast)
               +- Relation [track_id#1893,genre#1894,label#1895] parquet

== Analyzed Logical Plan ==
genre: string, device: strin

In [ ]:
+----------+-------------------------+---------------+------------------+
| Stage ID | Operations              | Tasks Count   | Shuffle Size     |
+----------+-------------------------+---------------+------------------+
| Stage 0  | Read events             | 8             | Shuffle Write:   |
|          | Filter (completed=True) | 8             | ~15-18 MB        |
|          | Filter (duration>30)    | 8             |                  |
+----------+-------------------------+---------------+------------------+
| Stage 1  | Broadcast tracks        | 8             | No Shuffle       |
|          | (tracks sent to all)    | 8             | (broadcast only) |
+----------+-------------------------+---------------+------------------+
| Stage 2  | Broadcast join          | 8             | Shuffle Write:   |
|          | groupBy genre,device,tier| 8            | ~12-15 MB        |
|          | count(*)                | 8             |                  |
|          | sum(revenue)            | 8             |                  |
|          | avg(duration)           | 8             |                  |
|          | countDistinct(user_id)  | 8             |                  |
+----------+-------------------------+---------------+------------------+
| Stage 3  | Filter (plays>50)       | 8             | Shuffle Write:   |
|          | orderBy(total_rev.desc) | 8             | ~5-8 MB          |
|          | Take 20 (show)          | 8             |                  |
+----------+-------------------------+---------------+------------------+

In [ ]:
# Stage Cost Analysis
+---------+---------------------+------------------+---------------------+
| Stage   | Primary Cost Factor | Approximate Time | % of Total Time     |
+---------+---------------------+------------------+---------------------+
| Stage 0 | I/O + filtering     | Low              | 10-15%              |
| Stage 1 | Network broadcast   | Low              | 5-10%               |
| Stage 2 | CPU + Shuffle +     | High             | 60-70%              |
|         | countDistinct       |                  |                     |
| Stage 3 | Sorting + Shuffle   | Medium           | 15-20%              |
+---------+---------------------+------------------+---------------------+

In [ ]:
# Optimization Impact Table
+------------------------+------------------+---------------------+
| Optimization           | Before           | After (projected)   |
+------------------------+------------------+---------------------+
| Original               | Stage 2: 60-70%  | -                   |
| Use approx_count_distinct | Stage 2 heavy | Stage 2: 40-50%    |
| Filter before join     | 800k events      | ~400k events        |
| Partition tuning       | 8 partitions     | 16 partitions       |
| Projected speedup      | -                | 25-35%              |
+------------------------+------------------+---------------------+

In [ ]:
## Task 6
Step-by-Step Flow
You write Python code → SparkSession receives transformations and actions

Catalyst optimizer creates and optimizes a logical plan (predicate pushdown, column pruning, constant folding)

Physical plan is generated with specific execution strategies (join selection, aggregation algorithms)

DAG Scheduler breaks the plan into stages at shuffle boundaries

Task Scheduler assigns tasks (one per partition per stage) to available executors

Executors run tasks in parallel across the cluster and return results to driver

Results are returned to the user or written to storage
==================================================================
Key Rules
=======================================================
1 action = 1 job (count(), show(), save(), collect(), etc.)

1 shuffle = 1 stage boundary (groupBy, join, distinct, orderBy, repartition)

Stages = number of shuffles + 1 (base stage for reading)

Tasks per stage = number of partitions at that stage's input

Broadcast join = NO shuffle (if small table <10MB or explicitly hinted)

SortMerge join = 2 shuffles (both tables repartitioned by join key)

Narrow transformations = No shuffle (filter, map, select, withColumn)

Wide transformations = Shuffle required (groupBy, join, distinct)

# Reference examples - see executive mapping tables in the Tasks 1-5 above



In [ ]:
## Query Planning Checklist
+----------------------------------+----------------+-----------------------------------+
| Checkpoint                       | Status (✓/✗)   | Notes                             |
+----------------------------------+----------------+-----------------------------------+
| Use .explain() before running    | [ ]            | Predict stage count & shuffles    |
| Check logical plan for pushdown  | [ ]            | Verify filters pushed to source   |
| Verify column pruning            | [ ]            | Select only needed columns        |
| Check for hidden shuffles        | [ ]            | distinct, orderBy, union, etc.    |
| Review physical plan strategy    | [ ]            | Confirm join/agg algorithms       |
| Analyze partition pruning        | [ ]            | For partitioned tables             |
+----------------------------------+----------------+-----------------------------------+

## Join Optimization Checklist
+----------------------------------+----------------+-----------------------------------+
| Checkpoint                       | Status (✓/✗)   | Notes                             |
+----------------------------------+----------------+-----------------------------------+
| Use broadcast for small tables   | [ ]            | Tables <10MB after filtering      |
| Add explicit broadcast hint      | [ ]            | join(broadcast(df), "key")        |
| Filter before join               | [ ]            | Reduce shuffle size               |
| Consider bucketing               | [ ]            | For repeated joins on same key    |
| Check join key data types        | [ ]            | Avoid implicit casts              |
| Verify join condition selectivity| [ ]            | Ensure it's not Cartesian         |
| Monitor join side selection      | [ ]            | Build vs probe side optimization  |
+----------------------------------+----------------+-----------------------------------+

## Caching Strategy Checklist
+----------------------------------+----------------+-----------------------------------+
| Checkpoint                       | Status (✓/✗)   | Notes                             |
+----------------------------------+----------------+-----------------------------------+
| Cache reused DataFrames          | [ ]            | Used across multiple actions      |
| Unpersist when done              | [ ]            | Free memory immediately           |
| Check Storage tab verification   | [ ]            | Confirm data actually cached      |
| Choose cache format wisely       | [ ]            | MEMORY_ONLY vs MEMORY_AND_DISK    |
| Monitor cache memory usage       | [ ]            | Avoid excessive GC                |
| Consider lazy caching            | [ ]            | cache() vs persist() trade-offs   |
| Check for over-caching           | [ ]            | Don't cache if used once          |
+----------------------------------+----------------+-----------------------------------+

## Shuffle Minimization Checklist
+----------------------------------+----------------+-----------------------------------+
| Checkpoint                       | Status (✓/✗)   | Notes                             |
+----------------------------------+----------------+-----------------------------------+
| Combine multiple aggregations    | [ ]            | Single groupBy vs multiple        |
| Filter early in pipeline         | [ ]            | Reduce data before shuffle        |
| Use approximate functions        | [ ]            | approx_count_distinct when possible|
| Consider partition pruning       | [ ]            | For date-partitioned tables       |
| Reduce shuffle partitions        | [ ]            | Tune spark.sql.shuffle.partitions |
| Use map-side combine             | [ ]            | Enable for aggregations           |
| Avoid distinct when possible     | [ ]            | Use groupBy + count instead       |
| Check for multiple shuffles      | [ ]            | Chain of wide transformations     |
+----------------------------------+----------------+-----------------------------------+

## Monitoring Checklist
+----------------------------------+----------------+-----------------------------------+
| Checkpoint                       | Status (✓/✗)   | Notes                             |
+----------------------------------+----------------+-----------------------------------+
| Check Stages tab for skew        | [ ]            | Tasks with widely varying times   |
| Monitor shuffle spill            | [ ]            | Spill (Memory) and Spill (Disk)   |
| Watch GC time                    | [ ]            | Should be <10% of task time       |
| Review executor memory usage     | [ ]            | Executors tab metrics             |
| Check task duration distribution | [ ]            | Min/Med/Max comparison            |
| Monitor scheduler delay          | [ ]            | Indicates cluster load            |
| Review shuffle read/write sizes  | [ ]            | Compare to expected data volume   |
| Check for straggler tasks        | [ ]            | Speculative execution may help    |
+----------------------------------+----------------+-----------------------------------+

## Partition Tuning Checklist
+----------------------------------+----------------+-----------------------------------+
| Checkpoint                       | Status (✓/✗)   | Notes                             |
+----------------------------------+----------------+-----------------------------------+
| Set appropriate shuffle partitions| [ ]           | Default 200 often too high        |
| Avoid too few partitions         | [ ]            | Underutilized cluster             |
| Avoid too many partitions        | [ ]            | Task scheduling overhead          |
| Consider custom partitioning     | [ ]            | For known data patterns           |
| Check input partition count      | [ ]            | Source file splitting             |
| Monitor partition sizes          | [ ]            | Aim for ~128MB per partition      |
| Use coalesce() after filter      | [ ]            | Reduce partitions when data shrinks|
| Use repartition() for balance    | [ ]            | Fix data skew                      |
+----------------------------------+----------------+-----------------------------------+

## Code Patterns to Avoid Checklist
+----------------------------------+----------------+-----------------------------------+
| Checkpoint                       | Status (✓/✗)   | Notes                             |
+----------------------------------+----------------+-----------------------------------+
| No Cartesian joins               | [ ]            | Missing join conditions           |
| Avoid collect() on large data    | [ ]            | Driver memory overflow risk       |
| Don't use repartition(1)         | [ ]            | Creates single file bottleneck    |
| Avoid row-by-row operations      | [ ]            | Use DataFrame API over UDFs       |
| No nested loops in Spark         | [ ]            | Use joins or window functions     |
| Avoid count() before write       | [ ]            | Unnecessary job execution         |
| Don't use df.collect().foreach   | [ ]            | Use df.foreach() instead          |
| Avoid unnecessary caching        | [ ]            | Don't cache intermediate results  |
+----------------------------------+----------------+-----------------------------------+

## Quick Diagnostic Summary Table
+---------------------+---------------------+---------------------+
| Symptom             | Possible Cause      | Recommended Action  |
+---------------------+---------------------+---------------------+
| Many short tasks    | Too many partitions | Use coalesce()      |
| Few long tasks      | Data skew           | Salting/optimization|
| High shuffle write  | Large shuffle       | Filter earlier      |
| High GC time        | Memory pressure     | Increase memory/partitions |
| Task failures       | OOM errors          | Increase partitions |
| Slow broadcast      | Large broadcast     | Check threshold     |
| Spill to disk       | Insufficient memory | Increase partitions |
| Stages > expected   | Hidden shuffles     | Check plan with explain()|
+---------------------+---------------------+---------------------+